# Clase 218 — Content-based: TF-IDF + sentence-transformers + FAISS

Movies sintéticas con título + overview + genres. Recomendamos por similitud de contenido.

Requiere: `pip install scikit-learn sentence-transformers faiss-cpu`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.DataFrame([
    {'id': 1,  'title': 'Toy Story',           'overview': 'Cowboy doll and astronaut toy adventures friendship.', 'genres': 'animation children comedy'},
    {'id': 2,  'title': 'Jumanji',             'overview': 'Magical board game jungle wild animals.',              'genres': 'adventure children fantasy'},
    {'id': 3,  'title': 'Heat',                'overview': 'Bank heist crew detective pursuit Los Angeles.',       'genres': 'action crime thriller'},
    {'id': 4,  'title': 'Goldeneye',           'overview': 'British spy secret agent satellite weapon villain.',   'genres': 'action adventure thriller'},
    {'id': 5,  'title': 'The Lion King',       'overview': 'Lion cub father betrayed uncle Africa savanna.',       'genres': 'animation adventure drama'},
    {'id': 6,  'title': 'Pulp Fiction',        'overview': 'Hitmen boxer gangster intersecting stories LA.',       'genres': 'crime drama thriller'},
    {'id': 7,  'title': 'Finding Nemo',        'overview': 'Clownfish ocean adventure son rescue father.',         'genres': 'animation children adventure'},
    {'id': 8,  'title': 'The Matrix',          'overview': 'Hacker discovers reality simulation rebel against AI.',
                                                                                                                     'genres': 'action sci-fi thriller'},
    {'id': 9,  'title': 'Forrest Gump',        'overview': 'Slow-witted man witnesses 20th century history love.', 'genres': 'comedy drama romance'},
    {'id': 10, 'title': 'Inception',           'overview': 'Dream thief mind heist subconscious layers.',          'genres': 'action sci-fi thriller'},
    {'id': 11, 'title': 'Shrek',               'overview': 'Ogre swamp princess rescue donkey fairy tale.',        'genres': 'animation comedy adventure'},
    {'id': 12, 'title': 'The Dark Knight',     'overview': 'Batman Joker chaos Gotham villain moral.',             'genres': 'action crime thriller'},
])

movies['text'] = movies.title + ' ' + movies.overview + ' ' + movies.genres
print(movies[['id', 'title']].to_string(index=False))

## 1. TF-IDF + item similarity

In [ ]:
vec = TfidfVectorizer(max_features=200, ngram_range=(1, 2), stop_words='english')
M = vec.fit_transform(movies.text)
print(f'TF-IDF shape: {M.shape}')

sim = cosine_similarity(M)

def top_similar_tfidf(title, n=5):
    i = movies.index[movies.title == title][0]
    scores = sim[i].copy(); scores[i] = -1
    top = np.argsort(-scores)[:n]
    return [(movies.iloc[j].title, round(scores[j], 3)) for j in top]

for t in ['Toy Story', 'Heat', 'The Matrix']:
    print(f'\nSimilar a "{t}":')
    for name, s in top_similar_tfidf(t): print(f'  {s:.3f}  {name}')

## 2. User profile + recomendación

In [ ]:
# user_42 le gustaron 3 animaciones (ratings ≥ 4)
user_ratings = {1: 5, 5: 4, 7: 5}   # Toy Story, Lion King, Finding Nemo

from scipy.sparse import csr_matrix
weights = np.array([user_ratings.get(mid, 0) for mid in movies.id])
mask = weights > 0
user_profile = (M[mask].multiply(weights[mask][:, None])).sum(axis=0) / weights[mask].sum()
user_profile = np.asarray(user_profile).ravel()

scores = M @ user_profile
# Excluir las que ya vio
scores[mask] = -1
top = np.argsort(-scores)[:5]
print('Recomendaciones para user que le gustaron Toy Story, Lion King, Finding Nemo:')
for j in top:
    print(f'  {scores[j]:.3f}  {movies.iloc[j].title}  ({movies.iloc[j].genres})')

## 3. Embeddings semánticos con sentence-transformers

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer('all-MiniLM-L6-v2')
    emb = st.encode(movies.text.tolist(), normalize_embeddings=True)
    print('embeddings shape:', emb.shape)

    def top_similar_emb(title, n=5):
        i = movies.index[movies.title == title][0]
        s = emb @ emb[i]; s[i] = -1
        top = np.argsort(-s)[:n]
        return [(movies.iloc[j].title, round(float(s[j]), 3)) for j in top]

    for t in ['Toy Story', 'The Matrix']:
        print(f'\nSimilar a "{t}" (embeddings):')
        for name, s in top_similar_emb(t): print(f'  {s:.3f}  {name}')
        print(f'  vs TF-IDF:')
        for name, s in top_similar_tfidf(t): print(f'  {s:.3f}  {name}')
except ImportError:
    print('pip install sentence-transformers para esta celda')

## 4. FAISS para retrieval rápido (escala a millones)

In [ ]:
try:
    import faiss, time
    d = emb.shape[1]
    index = faiss.IndexFlatIP(d)   # inner product (= cosine si normalizamos)
    index.add(emb.astype('float32'))
    print(f'index size: {index.ntotal} items')

    user_profile_emb = emb[[0, 4, 6]].mean(axis=0)   # gusto = avg de Toy Story + Lion King + Nemo
    user_profile_emb = user_profile_emb / np.linalg.norm(user_profile_emb)

    t0 = time.perf_counter()
    D, I = index.search(user_profile_emb.astype('float32').reshape(1, -1), k=5)
    print(f'FAISS top-5 search: {(time.perf_counter() - t0) * 1000:.2f} ms')
    for j, s in zip(I[0], D[0]):
        print(f'  {s:.3f}  {movies.iloc[j].title}')
except (ImportError, NameError):
    print('pip install faiss-cpu (y sentence-transformers para emb) para esta celda')

## Ejercicio guiado

1. Bajá MovieLens + sinopsis (TMDB API o Kaggle). Repetí con 10K movies reales.
2. Comparé top-10 TF-IDF vs sentence-transformers vs CF (Clase 216) para el mismo user.
3. Coverage: ¿qué % del catálogo es recomendado al menos una vez para 1000 users distintos?
4. MMR (Maximal Marginal Relevance) para diversidad: penalizar items similares a los ya seleccionados.
5. Cold-start demo: agregá una movie nueva (sin ratings). CF la ignora; content-based la recomienda.

## Conclusiones

- Content-based brilla en cold-start de items y dominios con metadata rica.
- TF-IDF sigue siendo baseline sólido; embeddings añaden semántica.
- FAISS hace retrieval factible a escala de millones de items.
- Solo content-based = filter bubble; combinar con CF (Clase 219).

## ✅ Soluciones de los ejercicios

Content-based no necesita historial de otros usuarios: recomienda por **parecido de
contenido**. Usamos TF-IDF (`sklearn`) sobre un catálogo sintético. `sentence-transformers`
y `faiss` no están instalados, así que el paso "embeddings modernos" lo hacemos con **LSA
(TruncatedSVD sobre TF-IDF)** como stand-in semántico y el "FAISS" con búsqueda por producto
interno en numpy. Sin internet.

### Catálogo sintético

Películas con `title + overview + genres`. Construimos texto por película para vectorizar.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

temas = {
    "animacion": "toys friendship colorful family adventure kids playful",
    "accion":    "explosions hero fight chase guns mission war soldier",
    "romance":   "love couple wedding heartbreak kiss relationship emotional",
    "terror":    "ghost haunted blood scary night killer fear dark",
    "scifi":     "space robot future alien spaceship technology galaxy",
}
titulos = {"animacion": "Toy", "accion": "Strike", "romance": "Hearts",
           "terror": "Haunt", "scifi": "Orbit"}
movies = []
for g, words in temas.items():
    for j in range(8):
        ws = words.split()
        overview = " ".join(rng.choice(ws, size=6))
        movies.append({"movie_id": len(movies),
                       "title": f"{titulos[g]} {j+1}",
                       "genre": g,
                       "text": f"{titulos[g]} {j+1} {overview} {g}"})
n_movies = len(movies)
print("películas:", n_movies, "| ejemplo:", movies[0]["title"], "->", movies[0]["genre"])
assert n_movies == 40
print("OK — catálogo sintético listo")

### Ejercicio 1 — TF-IDF base

`TfidfVectorizer(max_features=10000, ngram_range=(1,2))` sobre el texto de cada película.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [m["text"] for m in movies]
vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
tfidf = vec.fit_transform(corpus)
print("tfidf matrix shape:", tfidf.shape, "| vocab:", len(vec.vocabulary_))

assert tfidf.shape[0] == n_movies
assert tfidf.shape[1] <= 10000
print("OK ejercicio 1 — matriz TF-IDF construida")

### Ejercicio 2 — Similitud item-item

`cosine_similarity(tfidf)`. Para una película de animación, los top-10 más similares deben
ser otras animaciones (mismo vocabulario).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(tfidf)
np.fill_diagonal(sim, -np.inf)
base = 0                                        # "Toy 1" (animacion)
top10 = np.argsort(sim[base])[-10:][::-1]
generos = [movies[i]["genre"] for i in top10]
print(f"similares a '{movies[base]['title']}' ({movies[base]['genre']}):")
print("  géneros de los top-10:", generos)

same = sum(g == movies[base]["genre"] for g in generos)
assert same >= 6, "la mayoría de los similares comparten género"
print(f"OK ejercicio 2 — {same}/10 top similares son del mismo género (animación)")

### Ejercicio 3 — Perfil de usuario

Tomamos los items que el user calificó ≥4 y promediamos sus perfiles TF-IDF (ponderado por
rating). Recomendamos los top-10 más cercanos a ese perfil, excluyendo lo ya visto.

In [ ]:
# user al que le gusta el terror: califica alto pelis de terror
liked = [m["movie_id"] for m in movies if m["genre"] == "terror"][:5]
ratings = {mid: 5 for mid in liked}
tfidf_arr = tfidf.toarray()

weights = np.array([ratings[m] for m in liked])
user_profile = (tfidf_arr[liked] * weights[:, None]).sum(0) / weights.sum()

scores = cosine_similarity(user_profile.reshape(1, -1), tfidf_arr).ravel()
scores[liked] = -np.inf                          # excluir vistos
rec = np.argsort(scores)[-10:][::-1]
generos_rec = [movies[i]["genre"] for i in rec]
print("recomendaciones para el fan del terror:", generos_rec)

assert generos_rec.count("terror") >= 2, "el perfil recomienda más terror"
assert not set(rec) & set(liked)
print("OK ejercicio 3 — perfil de usuario construido y top-10 recomendado")

### Ejercicio 4 — "Embeddings modernos" (LSA como stand-in)

`SentenceTransformer` no está instalado. Como aproximación semántica usamos **LSA**
(`TruncatedSVD` sobre TF-IDF): comprime el vocabulario a factores latentes que agrupan
sinónimos/temas. Dejamos el código real de sentence-transformers como referencia.

In [ ]:
# --- Real (referencia) ------------------------------------------------------
# from sentence_transformers import SentenceTransformer
# emb = SentenceTransformer('all-MiniLM-L6-v2').encode([m['text'] for m in movies])
# ---------------------------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer
    emb = SentenceTransformer('all-MiniLM-L6-v2').encode(corpus)
    fuente = "sentence-transformers"
except Exception:
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize
    emb = normalize(TruncatedSVD(n_components=16, random_state=0).fit_transform(tfidf))
    fuente = "LSA (TruncatedSVD) stand-in"

print("embeddings de:", fuente, "| shape:", emb.shape)
sim_emb = cosine_similarity(emb)
np.fill_diagonal(sim_emb, -np.inf)
vecino_emb = int(np.argmax(sim_emb[base]))
print(f"vecino semántico de '{movies[base]['title']}': '{movies[vecino_emb]['title']}' "
      f"({movies[vecino_emb]['genre']})")

assert emb.shape[0] == n_movies
assert movies[vecino_emb]["genre"] == movies[base]["genre"]
print("OK ejercicio 4 — embeddings (LSA) capturan temas; vecino del mismo género")

### Ejercicio 5 — "FAISS rápido" (búsqueda por producto interno)

`faiss.IndexFlatIP` hace búsqueda exacta por producto interno (= coseno si los vectores están
normalizados). Sin faiss, lo replicamos con un producto matriz-vector en numpy y `argsort`.

In [ ]:
# --- Real (referencia) ------------------------------------------------------
# import faiss; index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
# D, I = index.search(user_vec, k=10)
# ---------------------------------------------------------------------------
def flat_ip_search(index_matrix, query, k=10):
    scores = index_matrix @ query                # producto interno con todos los items
    topk = np.argsort(scores)[-k:][::-1]
    return scores[topk], topk

# perfil del fan de scifi en el espacio de embeddings
scifi_ids = [m["movie_id"] for m in movies if m["genre"] == "scifi"][:5]
q = emb[scifi_ids].mean(0)
q = q / (np.linalg.norm(q) + 1e-9)
D, I = flat_ip_search(emb, q, k=10)
print("top-10 por 'FAISS' (IP):", I.tolist())
print("géneros:", [movies[i]["genre"] for i in I])

assert len(I) == 10
assert [movies[i]["genre"] for i in I].count("scifi") >= 4
print("OK ejercicio 5 — búsqueda top-k por producto interno (equivalente a IndexFlatIP)")